# T03: Multimodales RAG mit Text und Bildern

Eine RAG-Pipeline, die aus einem PDF sowohl Text als auch Bilder findet:

1. Das PDF in Text- und Bild-Chunks zerlegen
2. Text mit OpenAI embedden, Bilder mit einem Vision-Language-Modell
3. Beides in getrennten ChromaDB-Collections speichern
4. Mit einem kombinierten Retriever aus beiden suchen
5. Text und Bilder an das LLM übergeben, für eine belegte Antwort

Code, Prompts und Dokumente sind auf Englisch (Begründung in T01).

> **Optional, nicht Teil des Workshops.** Abschnitte 1 und 2 laufen überall. Ab Abschnitt 3 werden Bilder mit `Qwen/Qwen3-VL-Embedding-2B` embedded; das Modell lädt etwa 4 GB herunter und braucht rund 8 GB freien Arbeitsspeicher. In einer Session mit ~7 GB stürzt der Kernel dort wahrscheinlich ab; dafür einen Rechner mit mehr Speicher verwenden.

## Setup

Benötigt `OPENAI_API_KEY` (lokal: in `.env.local` im Hauptverzeichnis des Repos).

In [ ]:
import base64
import io
import sys

import matplotlib.pyplot as plt
from loguru import logger

from conversational_toolkit.agents.rag import RAG
from conversational_toolkit.chunking.base import Chunk
from conversational_toolkit.chunking.pdf_chunker import PDFChunker
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.llms.base import LLMMessage, MessageContent, Roles
from conversational_toolkit.retriever.vectorstore_retriever import (
    CompositeVectorStoreRetriever,
    VectorStoreRetriever,
)
from ki_kompass_rag.baseline_rag import (
    DATA_DIR,
    DB_DIR,
    EMBEDDING_MODEL,
    ask,
    build_llm,
    build_vector_store,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")


def show_image(chunk: Chunk, size: int = 5) -> None:
    image_format = chunk.mime_type.split("/")[1]
    image = plt.imread(io.BytesIO(base64.b64decode(chunk.content)), format=image_format)
    plt.figure(figsize=(size, size))
    plt.imshow(image)
    plt.axis("off")
    plt.show()

## 1. Das Dokument zerlegen

`PDFChunker.make_chunks()` liefert einen Text-Chunk pro Abschnitt und standardmässig einen Chunk pro eingebettetem Bild. Bilder werden als base64-kodiertes PNG in `content` gespeichert und passen so in dieselbe `Chunk`-Struktur.

In [ ]:
chunks = PDFChunker(do_ocr=False).make_chunks(
    str(DATA_DIR / "EPD_cardboard_redbox_cartonpallet.pdf")
)
text_chunks = [c for c in chunks if c.mime_type.startswith("text/")]
image_chunks = [c for c in chunks if c.mime_type.startswith("image/")]
print(f"{len(text_chunks)} text chunks, {len(image_chunks)} images")

for chunk in image_chunks:
    print(chunk.title)
    show_image(chunk, size=3)

## 2. Text embedden

Text-Chunks kommen in eine eigene Collection, embedded mit `text-embedding-3-small` wie in T01.

In [ ]:
text_embedding_model = OpenAIEmbeddings(EMBEDDING_MODEL)
text_vector_store = await build_vector_store(
    text_chunks, text_embedding_model, db_path=DB_DIR / "t03_text", reset=True
)

query = "What is the amount of PBT/vPvB substances?"
results = await VectorStoreRetriever(
    text_embedding_model, text_vector_store, top_k=3
).retrieve(query)
for r in results:
    print(f"{r.score:.3f}  {r.title}")
print(f"\n{results[0].content}")

## 3. Bilder embedden

> Braucht etwa 8 GB freien Arbeitsspeicher. Die Abschnitte 3 bis 6 hängen von diesem Modell ab.

`Qwen3VLEmbeddings` bildet Bilder und Text in denselben Vektorraum ab, sodass Bilder mit einer Textfrage gefunden werden: `get_embeddings` kodiert die Bild-Chunks, `get_text_embeddings` die Frage.

In [ ]:
from conversational_toolkit.embeddings.qwen_vl import Qwen3VLEmbeddings

image_embedding_model = Qwen3VLEmbeddings()
image_vector_store = await build_vector_store(
    image_chunks, image_embedding_model, db_path=DB_DIR / "t03_image", reset=True
)

In [ ]:
query = "The logo of a company"  # or "A carton box on a table", "The EPD logo"

query_embedding = await image_embedding_model.get_text_embeddings(query)
for r in await image_vector_store.get_chunks_by_embedding(query_embedding[0], top_k=3):
    print(f"{r.score:.3f}  {r.title}")
    show_image(r, size=4)

## 4. Kombiniertes Retrieval

`CompositeVectorStoreRetriever` fragt beide Stores parallel ab und hängt die Ergebnisse aneinander: hier die 3 besten Text-Chunks und das beste Bild.

In [ ]:
composite_retriever = CompositeVectorStoreRetriever(
    embedding_models=[text_embedding_model, image_embedding_model],
    vector_stores=[text_vector_store, image_vector_store],
    top_k=[3, 1],
)

results = await composite_retriever.retrieve(
    "What is the amount of PBT/vPvB substances?"
)
for r in results:
    print(f"{r.mime_type}  {r.title}")
    if r.mime_type.startswith("image/"):
        show_image(r)

## 5. Ein Bild an das LLM senden

Eine multimodale Nachricht enthält einen Textblock und einen Bildblock (base64-PNG). `generate` liefert die ganze Antwort, `generate_stream` gibt sie Stück für Stück aus.

In [ ]:
llm = build_llm("openai")
messages = [
    LLMMessage(
        role=Roles.USER,
        content=[
            MessageContent(type="text", text="What is in this image?"),
            MessageContent(type="image", image_url=results[-1].content),
        ],
    )
]

answer = await llm.generate(messages)
print(answer.content[0].text)

In [ ]:
async for chunk in llm.generate_stream(messages):
    print("".join(c.text or "" for c in chunk.content), end="")

## 6. Vollständige RAG-Pipeline

Der `RAG`-Agent mit dem kombinierten Retriever schickt die gefundenen Texte und Bilder an das LLM. `ask()` zeigt die Antwort und die Quellen; gefundene Bilder werden darunter angezeigt.

System-Prompt übersetzt:

> Beantworte Fragen zum Dokument nur mit den gefundenen Texten und Bildern. Wenn sie die Antwort nicht enthalten, sage, dass du es nicht weisst.

In [ ]:
agent = RAG(
    llm=llm,
    utility_llm=llm,
    system_prompt=(
        "Answer questions about the document using only the retrieved text and "
        "images. If they do not contain the answer, say you don't know."
    ),
    retrievers=[composite_retriever],
)

for query in [
    "What is the amount of PBT/vPvB substances? Short answer.",
    "What is the percentage of glue in the product? Short answer.",
]:
    print(f"\n>>> {query}")
    answer = await ask(agent, query)
    for source in answer.sources:
        if source.mime_type.startswith("image/"):
            show_image(source, size=4)